# ChArUco Camera Calibration

A step-by-step Jupyter notebook for calibrating a camera using a **ChArUco board**.

OpenCV ≥ 4.7, Python ≥ 3.8

---

### What you'll need
1. A printed ChArUco board (generate from [ChArUco Board Generator](https://charuco-board-generator.vercel.app/))
2. A camera to calibrate
3. 10–15 photos of the board from different angles

---

## 1. Import Libraries

In [ ]:
import numpy as np
import cv2
import glob
import json
import os
import matplotlib.pyplot as plt
from pathlib import Path

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version  : {np.__version__}")

## 2. Configuration

Set the board parameters and image path. **Important:** measure the actual printed square size with a ruler!

In [ ]:
# -------------------------------------------------
#  EDIT THESE VALUES TO MATCH YOUR SETUP
# -------------------------------------------------

IMAGE_DIR = "./calib_images/"          # Folder containing calibration images
SQUARES_X = 7                            # Number of squares in X direction
SQUARES_Y = 5                            # Number of squares in Y direction
SQUARE_LENGTH_MM = 25.0                  # Measured square side length (mm)
MARKER_LENGTH_MM = None                  # Marker length (mm), None = 0.6 * square
ARUCO_DICT = "DICT_6X6_250"             # ArUco dictionary
FIX_PRINCIPAL_POINT = False              # Fix principal point at image centre
OUTPUT_DIR = "calibration_results"      # Output directory
SAMPLE_IMAGE = None                      # Optional: path to undistort a sample

# -------------------------------------------------

# Derived parameters
square_length = SQUARE_LENGTH_MM / 1000.0   # Convert mm to metres
marker_length = MARKER_LENGTH_MM
if marker_length is None:
    marker_length = square_length * 0.6
else:
    marker_length = marker_length / 1000.0

print(f"Board: {SQUARES_X}x{SQUARES_Y}")
print(f"Square size : {SQUARE_LENGTH_MM:.1f} mm")
print(f"Marker size : {marker_length*1000:.1f} mm")
print(f"Dictionary  : {ARUCO_DICT}")
print(f"Image dir   : {IMAGE_DIR}")

## 3. Helper: Resolve ArUco Dictionary

In [ ]:
def get_aruco_dict(name):
    """Resolve ArUco dictionary name to OpenCV object."""
    mapping = {
        "DICT_4X4_50": cv2.aruco.DICT_4X4_50,
        "DICT_4X4_100": cv2.aruco.DICT_4X4_100,
        "DICT_4X4_250": cv2.aruco.DICT_4X4_250,
        "DICT_4X4_1000": cv2.aruco.DICT_4X4_1000,
        "DICT_5X5_50": cv2.aruco.DICT_5X5_50,
        "DICT_5X5_100": cv2.aruco.DICT_5X5_100,
        "DICT_5X5_250": cv2.aruco.DICT_5X5_250,
        "DICT_5X5_1000": cv2.aruco.DICT_5X5_1000,
        "DICT_6X6_50": cv2.aruco.DICT_6X6_50,
        "DICT_6X6_100": cv2.aruco.DICT_6X6_100,
        "DICT_6X6_250": cv2.aruco.DICT_6X6_250,
        "DICT_6X6_1000": cv2.aruco.DICT_6X6_1000,
        "DICT_7X7_50": cv2.aruco.DICT_7X7_50,
        "DICT_7X7_100": cv2.aruco.DICT_7X7_100,
        "DICT_7X7_250": cv2.aruco.DICT_7X7_250,
        "DICT_7X7_1000": cv2.aruco.DICT_7X7_1000,
        "DICT_ARUCO_ORIGINAL": cv2.aruco.DICT_ARUCO_ORIGINAL,
    }
    if name not in mapping:
        print(f"Unknown dictionary '{name}', using DICT_6X6_250")
        name = "DICT_6X6_250"
    return cv2.aruco.getPredefinedDictionary(mapping[name])

## 4. Load Images

In [ ]:
def load_images(path_pattern):
    """Load sorted image paths from a directory or glob."""
    path = Path(path_pattern)
    if path.is_dir():
        patterns = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tiff"]
        files = []
        for p in patterns:
            files.extend(sorted(path.glob(p)))
            files.extend(sorted(path.glob(p.upper())))
    else:
        files = sorted(Path(".").parent.glob(path_pattern))
    
    if not files:
        raise FileNotFoundError(f"No images found at: {path_pattern}")
    print(f"Loaded {len(files)} image(s)")
    return files


images = load_images(IMAGE_DIR)
print("First 5 images:")
for p in images[:5]:
    print(f"  {p.name}")

## 5. Detect ChArUco Corners

This step detects ArUco markers in each image, then interpolates the chessboard corners from the marker positions.

In [ ]:
# Create board and dictionary
dictionary = get_aruco_dict(ARUCO_DICT)
board = cv2.aruco.CharucoBoard(
    (SQUARES_X, SQUARES_Y),
    square_length,
    marker_length,
    dictionary,
)

# Detect markers
all_corners = []
all_ids = []
valid_indices = []
detector_params = cv2.aruco.DetectorParameters()

for idx, img_path in enumerate(images):
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"  [SKIP] Cannot read: {img_path.name}")
        continue
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    marker_corners, marker_ids, _ = cv2.aruco.detectMarkers(
        gray, dictionary, parameters=detector_params
    )
    
    if marker_ids is None or len(marker_ids) < 4:
        print(f"  [SKIP] {img_path.name}: {0 if marker_ids is None else len(marker_ids)} markers")
        continue
    
    # Refine markers
    cv2.aruco.refineDetectedMarkers(gray, board, marker_corners, marker_ids)
    
    # Interpolate ChArUco corners
    charuco_corners, charuco_ids = cv2.aruco.interpolateCornersCharuco(
        marker_corners, marker_ids, gray, board
    )
    
    if charuco_ids is None or len(charuco_ids) < 4:
        print(f"  [SKIP] {img_path.name}: {0 if charuco_ids is None else len(charuco_ids)} corners")
        continue
    
    all_corners.append(charuco_corners)
    all_ids.append(charuco_ids)
    valid_indices.append(idx)
    print(f"  [OK]   {img_path.name}: {len(charuco_ids)} corners, {len(marker_ids)} markers")

print(f"\nTotal valid views: {len(all_corners)}")
assert len(all_corners) >= 5, f"Need >= 5 valid views, got {len(all_corners)}"

## 6. Visualise Detected Corners

Let's look at the first valid image to confirm detection quality.

In [ ]:
# Show the first valid image with detected corners
first_idx = valid_indices[0]
img = cv2.imread(str(images[first_idx]))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Draw detected ChArUco corners
cv2.aruco.drawDetectedCornersCharuco(img_rgb, all_corners[0], all_ids[0])

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.title(f"Detected ChArUco Corners — {images[first_idx].name}")
plt.axis("off")
plt.show()

## 7. Run Calibration

Now we run `calibrateCameraCharuco()` to obtain the camera matrix and distortion coefficients.

In [ ]:
# Get image size from first valid image
first_img = cv2.imread(str(images[valid_indices[0]]))
image_size = (first_img.shape[1], first_img.shape[0])
print(f"Image size: {image_size[0]} x {image_size[1]}")

# Calibration flags
flags = 0
if FIX_PRINCIPAL_POINT:
    flags |= cv2.CALIB_FIX_PRINCIPAL_POINT

# Run
print(f"Running calibration with {len(all_corners)} views...")
ret, mtx, dist, rvecs, tvecs = cv2.aruco.calibrateCameraCharuco(
    all_corners, all_ids, board, image_size, None, None, flags=flags
)
print(f"\nRMS reprojection error: {ret:.4f} pixels")

## 8. Results Summary

In [ ]:
fx, fy = mtx[0, 0], mtx[1, 1]
cx, cy = mtx[0, 2], mtx[1, 2]

print("Camera Matrix:")
print(mtx)
print(f"\nDistortion Coefficients: {dist.ravel()}")
print(f"\nFocal Length   : fx = {fx:.2f}, fy = {fy:.2f}")
print(f"Principal Point: cx = {cx:.2f}, cy = {cy:.2f}")
print(f"Image Size     : {image_size[0]} x {image_size[1]}")

# Sanity checks
print("\n--- Sanity Checks ---")
if image_size[0] * 0.5 < fx < image_size[0] * 5:
    print("  [OK] Focal length X looks reasonable")
else:
    print("  [WARN] Focal length X seems unusual")
if abs(cx - image_size[0] / 2) < image_size[0] * 0.2:
    print("  [OK] Principal point X near centre")
else:
    print("  [WARN] Principal point X far from centre")
if abs(cy - image_size[1] / 2) < image_size[1] * 0.2:
    print("  [OK] Principal point Y near centre")
else:
    print("  [WARN] Principal point Y far from centre")
if ret < 0.3:
    print("  [OK] RMS error is excellent (< 0.3 px)")
elif ret < 0.5:
    print("  [OK] RMS error is good (< 0.5 px)")
elif ret < 0.8:
    print("  [OK] RMS error is acceptable (< 0.8 px)")
else:
    print("  [WARN] RMS error is high (>= 0.8 px)")

## 9. Per-View Reprojection Errors

High error in a particular view may indicate motion blur, poor lighting, or an inaccurate board.

In [ ]:
def compute_per_view_errors(all_corners, all_ids, rvecs, tvecs, mtx, dist, board):
    """Compute reprojection error for each view."""
    errors = []
    for i in range(len(all_corners)):
        img_pts = all_corners[i].reshape(-1, 2)
        _, obj_pts, _ = board.matchImagePoints(all_corners[i], all_ids[i])
        proj, _ = cv2.projectPoints(obj_pts, rvecs[i], tvecs[i], mtx, dist)
        proj = proj.reshape(-1, 2)
        err = float(np.sqrt(np.mean(np.sum((img_pts - proj) ** 2, axis=1))))
        errors.append(err)
    return errors


errors = compute_per_view_errors(all_corners, all_ids, rvecs, tvecs, mtx, dist, board)

format_header = '{:>4s}  {:30s}  {:>8s}'
print(format_header.format('Idx', 'File', 'Error'))
print(format_header.format('-' * 4, '-' * 30, '-' * 8))
for i, (orig_idx, err) in enumerate(zip(valid_indices, errors)):
    warn = ' <- HIGH' if err > 0.8 else ''
    print(f"[{i:2d}]  {images[orig_idx].name:30s}  {err:.4f}{warn}")

## 10. Visualise Per-View Errors

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(range(len(errors)), errors)
plt.axhline(y=0.5, color='g', linestyle='--', label='Good (0.5 px)')
plt.axhline(y=0.8, color='r', linestyle='--', label='Acceptable (0.8 px)')
plt.xlabel("View index")
plt.ylabel("Reprojection error (px)")
plt.title("Per-View Reprojection Error")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 11. Save Results

Save as both JSON (human-readable) and OpenCV XML (for use with `cv.FileStorage`).

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# JSON
json_data = {
    "rms_error": float(ret),
    "image_size": list(image_size),
    "camera_matrix": mtx.tolist(),
    "distortion_coefficients": dist.tolist(),
}
json_path = out_dir / "calibration_results.json"
with open(json_path, "w") as f:
    json.dump(json_data, f, indent=2)
print(f"Saved: {json_path}")

# XML (OpenCV-compatible)
xml_path = out_dir / "calibration_results.xml"
fs = cv2.FileStorage(str(xml_path), cv2.FILE_STORAGE_WRITE)
fs.write("rms_error", ret)
fs.write("image_width", image_size[0])
fs.write("image_height", image_size[1])
fs.write("camera_matrix", mtx)
fs.write("distortion_coefficients", dist)
fs.release()
print(f"Saved: {xml_path}")

## 12. Save Visualisations

Draw detected corners and coordinate axes on each valid image.

In [ ]:
viz_dir = out_dir / "visualisations"
viz_dir.mkdir(parents=True, exist_ok=True)

for i, img_idx in enumerate(valid_indices):
    img = cv2.imread(str(images[img_idx]))
    if img is None:
        continue
    cv2.aruco.drawDetectedCornersCharuco(img, all_corners[i], all_ids[i])
    cv2.drawFrameAxes(img, mtx, dist, rvecs[i], tvecs[i], 0.03)
    out_file = viz_dir / f"frame_{img_idx:03d}.jpg"
    cv2.imwrite(str(out_file), img)

print(f"Saved {len(valid_indices)} visualisations to: {viz_dir}/")

## 13. Undistort a Sample Image (Optional)

If you provided a `SAMPLE_IMAGE` path above, this cell undistorts it using the calibration results.

In [ ]:
if SAMPLE_IMAGE is not None:
    sample = cv2.imread(SAMPLE_IMAGE)
    if sample is not None:
        h, w = sample.shape[:2]
        new_mtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), 1, (w, h))
        
        # Undistort
        undistorted = cv2.undistort(sample, mtx, dist, None, new_mtx)
        x, y, w_roi, h_roi = roi
        if w_roi > 0 and h_roi > 0:
            undistorted = undistorted[y:y+h_roi, x:x+w_roi]
        
        cv2.imwrite(str(out_dir / "undistorted_sample.jpg"), undistorted)
        
        # Show comparison
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original")
        axes[0].axis("off")
        axes[1].imshow(cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB))
        axes[1].set_title("Undistorted")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()
        print(f"Saved undistorted sample to: {out_dir / 'undistorted_sample.jpg'}")
    else:
        print(f"Could not read sample image: {SAMPLE_IMAGE}")
else:
    print("No sample image specified. Set SAMPLE_IMAGE in the config cell to enable.")

## 14. Using the Calibration

To use the saved calibration in your own code:

```python
import cv2
import numpy as np

# Load XML calibration
fs = cv2.FileStorage("calibration_results.xml", cv2.FILE_STORAGE_READ)
mtx = fs.getNode("camera_matrix").mat()
dist = fs.getNode("distortion_coefficients").mat()
fs.release()

# Undistort an image
img = cv2.imread("image.jpg")
undistorted = cv2.undistort(img, mtx, dist)
```

Or load from JSON:

```python
import json
import numpy as np

with open("calibration_results.json") as f:
    data = json.load(f)
mtx = np.array(data["camera_matrix"])
dist = np.array(data["distortion_coefficients"])
```

---

**Done!** Your camera is now calibrated.